In [0]:
%pip install --upgrade "databricks-sdk>=0.118.0" psycopg2-binary
dbutils.library.restartPython()

In [0]:
"""
Post-Deployment Validation Test Suite — Hi Genie Orchestrator
=============================================================

Runs after each app deployment to verify system readiness. Designed to be
executed as a Databricks Job for CI/CD validation (see
resources/post_deploy_tests.job.yml).

Unlike a log-scraping smoke test, this suite makes real authenticated HTTP
calls against the deployed app and asserts on actual response status/body —
it exercises the system, it does not infer health from OTel log substrings.

Test Plan:
  1. Health Check           — /api/health responds 200 with all modules operational.
  2. Negative Authorization — an unauthenticated request to /api/v1 is rejected
                               with 401 (proves requireVerifiedIdentity fails closed
                               rather than falling back to a shared identity).
  3. E2E Dispatch Flow      — plan -> publish -> orchestrate -> materialize ->
                               dispatch, asserting on each response body in turn.
  4. Negative Scoping       — a workstream_key that exists in one orchestration
                               is not found (404) when dispatched through a
                               DIFFERENT orchestration's URL, even though the key
                               string matches (proves orchestration_id binding).
  5. MCP tools/list         — the MCP endpoint responds and record_decision's
                               advertised schema requires workstream_id (matching
                               the DB NOT NULL contract on state.decisions).
  6. RLS Write Integrity    — calls start_session via MCP, verifies the returned
                               user_id matches the authenticated SPN identity
                               (proves withIdentity + RLS INSERT policy work E2E).
  7. RLS Policy Audit       — direct Lakebase pg_policies + pg_class query verifying
                               all 8 state tables have ENABLE + FORCE ROW LEVEL
                               SECURITY and all 16 expected policies exist.

Each test returns PASS/FAIL with diagnostic details. On failure, the full raw
request/response (not a truncated snippet) is included in the result so a
job failure is diagnosable without re-running the suite. The suite fails the
job if ANY critical test fails.
"""

import json
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient

# ── Configuration (overridable via job base_parameters / widgets) ───────────
# dbutils.widgets.getArgument(name, default) returns the widget's current
# value, or `default` if the widget doesn't exist — safe for both a Jobs run
# (resources/post_deploy_tests.job.yml's notebook_task.base_parameters
# auto-creates a widget per key) and an interactive/ad-hoc run with no
# widgets defined at all. dbutils.widgets.get(name) is NOT safe here: it
# raises InputWidgetNotDefined if the widget was never created, so it would
# need a getAll()-based existence check first — and dbutils.widgets.getAll()
# returns a dict[str, str] of widget name -> value (Scala: Map[String,
# String]), not an iterable of objects with a .name attribute, so iterating
# it as `for w in dbutils.widgets.getAll()` and reading `w.name` raises
# immediately (each `w` is already the string key).
def _widget(name: str, default: str) -> str:
    return dbutils.widgets.getArgument(name, default)


APP_NAME = _widget("app_name", "mcp-hi-genie-orchestrator-dev")
SECRET_SCOPE = _widget("secret_scope_name", "hi_genie_credentials")
CLIENT_ID_KEY = _widget("client_id_dbs_key", "client_id_dev_matthew_giglia_hi_genie")
CLIENT_SECRET_KEY = _widget("client_secret_dbs_key", "client_secret_dev_matthew_giglia_hi_genie")
TIMEOUT_SECONDS = 30
EXPECTED_MODULES = ["migrations", "telemetry", "memory", "dispatch", "mcp", "api"]

# Lakebase direct-query configuration (for RLS policy audit)
LAKEBASE_PROJECT_ID = _widget("lakebase_project_id", "hi-genie-orchestrator")
LAKEBASE_BRANCH = _widget("lakebase_branch", "development")
LAKEBASE_DATABASE = _widget("lakebase_database", "databricks_postgres")

print(f"[config] App: {APP_NAME}")
print(f"[config] Secret scope: {SECRET_SCOPE} (client_id key: {CLIENT_ID_KEY})")
print(f"[config] Lakebase: projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH}/databases/{LAKEBASE_DATABASE}")

In [0]:
@dataclass
class TestResult:
    """Individual test result."""
    name: str
    status: str  # PASS, FAIL, SKIP
    duration_ms: float = 0.0
    message: str = ""
    # Raw diagnostic detail (full request/response), never truncated — the
    # whole point of this suite is that a FAIL is diagnosable from the job
    # output alone, without needing to reproduce the failure interactively.
    diagnostic: dict = field(default_factory=dict)
    critical: bool = True  # If True, failure = job failure


class TestSuite:
    """Collects and reports test results."""

    def __init__(self, suite_name: str):
        self.suite_name = suite_name
        self.results: list[TestResult] = []
        self.start_time = time.time()

    def add(self, result: TestResult):
        self.results.append(result)
        icon = "PASS" if result.status == "PASS" else ("FAIL" if result.status == "FAIL" else "SKIP")
        print(f"  [{icon:>4}] {result.name} ({result.duration_ms:.0f}ms) {result.message}")
        if result.status == "FAIL" and result.diagnostic:
            # Preserve full diagnostic output for failures — never truncate.
            print(f"         diagnostic: {json.dumps(result.diagnostic, default=str)}")

    def summary(self) -> dict:
        total = len(self.results)
        passed = sum(1 for r in self.results if r.status == "PASS")
        failed = sum(1 for r in self.results if r.status == "FAIL")
        skipped = sum(1 for r in self.results if r.status == "SKIP")
        critical_failures = sum(1 for r in self.results if r.status == "FAIL" and r.critical)
        duration_s = time.time() - self.start_time

        return {
            "suite": self.suite_name,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "total": total,
            "passed": passed,
            "failed": failed,
            "skipped": skipped,
            "critical_failures": critical_failures,
            "duration_s": round(duration_s, 2),
            "verdict": "PASS" if critical_failures == 0 else "FAIL",
        }

    def assert_passed(self):
        """Raise if any critical test failed (fails the job)."""
        s = self.summary()
        print(f"\n{'=' * 60}")
        print(f"SUITE: {s['suite']} | VERDICT: {s['verdict']}")
        print(f"Total: {s['total']} | Passed: {s['passed']} | Failed: {s['failed']} | Skipped: {s['skipped']}")
        print(f"Duration: {s['duration_s']}s")
        print(f"{'=' * 60}")
        if s["critical_failures"] > 0:
            failing = [r for r in self.results if r.status == "FAIL" and r.critical]
            # Include full diagnostics in the raised message too, so the job's
            # failure notification (email/webhook) carries the raw evidence,
            # not just a one-line summary.
            details = "\n".join(
                f"  - {r.name}: {r.message}\n    diagnostic: {json.dumps(r.diagnostic, default=str)}"
                for r in failing
            )
            raise AssertionError(
                f"Post-deployment validation FAILED: {s['critical_failures']} critical failure(s):\n{details}"
            )


suite = TestSuite(f"post-deploy-{APP_NAME}")
print(f"\n[suite] Starting post-deployment validation for {APP_NAME}...\n")

In [0]:
import requests

w = WorkspaceClient()
WORKSPACE_HOST = w.config.host.rstrip("/")

app_info = w.apps.get(APP_NAME)
APP_URL = app_info.url.rstrip("/")
print(f"[client] App URL: {APP_URL}")

SPN_CLIENT_ID = dbutils.secrets.get(SECRET_SCOPE, CLIENT_ID_KEY)
SPN_CLIENT_SECRET = dbutils.secrets.get(SECRET_SCOPE, CLIENT_SECRET_KEY)

# Databricks Apps require workspace-authenticated traffic. The App SPN's
# client_credentials OAuth token IS accepted by the app's own HTTP ingress
# (see PROJECT_MEMORY Known Issue 10: wc.config.authenticate() is NOT valid
# for this — it mints a platform-REST-API-scoped token, not one the app's
# ingress accepts for a human/U2M identity path).
_token_resp = requests.post(
    f"{WORKSPACE_HOST}/oidc/v1/token",
    data={
        "grant_type": "client_credentials",
        "client_id": SPN_CLIENT_ID,
        "client_secret": SPN_CLIENT_SECRET,
        "scope": "all-apis",
    },
    timeout=TIMEOUT_SECONDS,
)
if _token_resp.status_code != 200:
    raise RuntimeError(f"SPN OAuth token acquisition failed: {_token_resp.status_code} {_token_resp.text}")
SPN_TOKEN = _token_resp.json()["access_token"]
print("[client] SPN OAuth token acquired")


def app_request(method: str, path: str, authenticated: bool = True, **kwargs) -> requests.Response:
    """Make a request to the deployed app. authenticated=False omits the
    Authorization header entirely, for negative-authorization tests."""
    url = f"{APP_URL}{path}"
    headers = kwargs.pop("headers", {})
    if authenticated:
        headers["Authorization"] = f"Bearer {SPN_TOKEN}"
    timeout = kwargs.pop("timeout", TIMEOUT_SECONDS)
    return requests.request(method, url, headers=headers, timeout=timeout, **kwargs)


print(f"[client] Authenticated HTTP client ready for {APP_NAME}")

In [0]:
print("─" * 40)
print("TEST 1: Health Check (/api/health)")
print("─" * 40)

start = time.time()
try:
    resp = app_request("GET", "/api/health")
    duration_ms = (time.time() - start) * 1000
    body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {"raw": resp.text}

    if resp.status_code != 200:
        suite.add(TestResult(
            name="health_check", status="FAIL", duration_ms=duration_ms,
            message=f"HTTP {resp.status_code}",
            diagnostic={"status_code": resp.status_code, "body": body},
        ))
    elif body.get("status") != "ok":
        suite.add(TestResult(
            name="health_check", status="FAIL", duration_ms=duration_ms,
            message=f"status is '{body.get('status')}', expected 'ok'",
            diagnostic={"body": body},
        ))
    else:
        modules = body.get("modules", {})
        missing_modules = [m for m in EXPECTED_MODULES if modules.get(m) is not True]
        if missing_modules:
            suite.add(TestResult(
                name="health_check", status="FAIL", duration_ms=duration_ms,
                message=f"modules not operational: {missing_modules}",
                diagnostic={"body": body},
            ))
        else:
            suite.add(TestResult(
                name="health_check", status="PASS", duration_ms=duration_ms,
                message=f"v{body.get('version', '?')} | all {len(EXPECTED_MODULES)} modules operational",
                diagnostic={"body": body},
            ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="health_check", status="FAIL", duration_ms=duration_ms,
        message=f"request failed: {type(e).__name__}: {e}",
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 2: Negative Authorization (no credentials)")
print("─" * 40)

start = time.time()
try:
    bogus_plan_id = str(uuid.uuid4())
    resp = app_request("GET", f"/api/v1/plans/{bogus_plan_id}", authenticated=False)
    duration_ms = (time.time() - start) * 1000
    body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {"raw": resp.text}

    if resp.status_code == 401:
        # The Databricks App auth sidecar returns 401 with an empty body when
        # the request has no valid credentials. The body format varies (may be
        # {"error": "UNAUTHENTICATED"} from our middleware, or {} from the
        # sidecar itself). Either is a valid rejection.
        suite.add(TestResult(
            name="negative_auth_unverified_identity", status="PASS", duration_ms=duration_ms,
            message="unauthenticated request correctly rejected with HTTP 401",
            diagnostic={"status_code": resp.status_code, "body": body},
        ))
    else:
        suite.add(TestResult(
            name="negative_auth_unverified_identity", status="FAIL", duration_ms=duration_ms,
            message=(
                f"expected HTTP 401, got {resp.status_code} — "
                "requireVerifiedIdentity may have regressed to a permissive fallback"
            ),
            diagnostic={"status_code": resp.status_code, "body": body},
        ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="negative_auth_unverified_identity", status="FAIL", duration_ms=duration_ms,
        message=f"request failed: {type(e).__name__}: {e}",
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 3: E2E Dispatch Flow")
print("─" * 40)

WORKSTREAM_KEY = "ws-post-deploy-check"
e2e_project_id = None
e2e_plan_id = None
e2e_snapshot_id = None
e2e_orchestration_id = None


def e2e_step(name: str, method: str, path: str, json_body: dict, expect_status: int, critical: bool = True):
    """Run one E2E step, add a TestResult, and return the parsed body (or None on failure)."""
    step_start = time.time()
    try:
        resp = app_request(method, path, json=json_body)
        duration_ms = (time.time() - step_start) * 1000
        body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {"raw": resp.text}

        if resp.status_code != expect_status:
            suite.add(TestResult(
                name=name, status="FAIL", duration_ms=duration_ms,
                message=f"{method} {path} → HTTP {resp.status_code}, expected {expect_status}",
                diagnostic={"request": json_body, "status_code": resp.status_code, "body": body},
                critical=critical,
            ))
            return None

        suite.add(TestResult(
            name=name, status="PASS", duration_ms=duration_ms,
            message=f"{method} {path} → HTTP {resp.status_code}",
            diagnostic={"body": body},
            critical=critical,
        ))
        return body
    except Exception as e:
        duration_ms = (time.time() - step_start) * 1000
        suite.add(TestResult(
            name=name, status="FAIL", duration_ms=duration_ms,
            message=f"{method} {path} raised {type(e).__name__}: {e}",
            diagnostic={"request": json_body},
            critical=critical,
        ))
        return None


# Step 0 — create project (required: migration 009 FK on dispatch.plans.project_id)
_body = e2e_step(
    "e2e_create_project", "POST", "/api/v1/projects",
    {"name": "Post-deploy E2E validation", "description": "Ephemeral project for post-deploy validation"},
    expect_status=201,
)
if _body:
    e2e_project_id = _body["project"]["project_id"]

# Step 1 — create plan (requires valid project_id from Step 0)
_body = e2e_step(
    "e2e_create_plan", "POST", "/api/v1/plans",
    {"project_id": e2e_project_id, "name": "Post-deploy E2E validation"},
    expect_status=201,
) if e2e_project_id else None
if _body:
    e2e_plan_id = _body["plan"]["plan_id"]

# Step 2 — publish a snapshot with one workstream
_body = e2e_step(
    "e2e_publish_plan", "POST", f"/api/v1/plans/{e2e_plan_id}/publish",
    {
        "content": {
            "workstreams": [{
                "workstream_key": WORKSTREAM_KEY,
                "name": "Post-deploy check workstream",
                "execution_tier": 0,
                "depends_on": [],
                "config": {},
            }],
        },
        "workstream_count": 1,
    },
    expect_status=201,
) if e2e_plan_id else None
if _body:
    e2e_snapshot_id = _body["snapshot"]["plan_snapshot_id"]

# Step 3 — create orchestration from the published snapshot
_body = e2e_step(
    "e2e_orchestrate", "POST", f"/api/v1/plans/{e2e_plan_id}/orchestrate",
    {"plan_snapshot_id": e2e_snapshot_id},
    expect_status=201,
) if e2e_plan_id and e2e_snapshot_id else None
if _body:
    e2e_orchestration_id = _body["orchestration"]["orchestration_id"]

# Step 4 — materialize workstreams from the snapshot into the orchestration
_body = e2e_step(
    "e2e_materialize", "POST", f"/api/v1/orchestrations/{e2e_orchestration_id}/materialize",
    {"plan_snapshot_id": e2e_snapshot_id},
    expect_status=201,
) if e2e_orchestration_id else None
if _body and _body.get("workstreams_materialized") != 1:
    suite.add(TestResult(
        name="e2e_materialize_count", status="FAIL", duration_ms=0,
        message=f"expected 1 workstream materialized, got {_body.get('workstreams_materialized')}",
        diagnostic={"body": _body},
    ))

# Step 5 — dispatch the materialized workstream
# worker_identity is now optional in DispatchSchema — the server defaults it to
# the authenticated caller's identity (from x-forwarded-user header). This avoids
# the SPN_CLIENT_ID vs sidecar-header identity mismatch that was causing RLS
# INSERT policy violations and crashing the app.
_body = e2e_step(
    "e2e_dispatch", "POST",
    f"/api/v1/orchestrations/{e2e_orchestration_id}/workstreams/{WORKSTREAM_KEY}/dispatch",
    {},
    expect_status=201,
) if e2e_orchestration_id else None
if _body and _body.get("status") != "IN_PROGRESS":
    suite.add(TestResult(
        name="e2e_dispatch_status", status="FAIL", duration_ms=0,
        message=f"expected workstream status IN_PROGRESS after dispatch, got {_body.get('status')}",
        diagnostic={"body": _body},
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 4: Negative Scoping (orchestration_id binding)")
print("─" * 40)

start = time.time()
try:
    if not e2e_orchestration_id:
        suite.add(TestResult(
            name="negative_scoping_cross_orchestration", status="SKIP", duration_ms=0,
            message="skipped — E2E flow did not produce an orchestration to compare against",
            critical=False,
        ))
    else:
        # Create a second, unrelated orchestration that has never had this
        # workstream_key materialized into it. Dispatching WORKSTREAM_KEY
        # through this orchestration's URL must 404 — not silently match
        # the row that actually belongs to e2e_orchestration_id, which would
        # indicate the dispatch/retry/pause/unpause handlers dropped their
        # orchestration_id predicate (the exact regression class this test
        # suite exists to catch).
        # Create a separate project first (migration 009 FK: plans.project_id → projects.project_id)
        other_project_resp = app_request(
            "POST", "/api/v1/projects",
            json={"name": "Negative-scoping control project", "description": "Ephemeral project for negative-scoping test"},
        )
        other_project_id = other_project_resp.json()["project"]["project_id"]

        other_plan_resp = app_request(
            "POST", "/api/v1/plans",
            json={"project_id": other_project_id, "name": "Post-deploy negative-scoping control plan"},
        )
        other_plan = other_plan_resp.json()["plan"]

        other_publish_resp = app_request(
            "POST", f"/api/v1/plans/{other_plan['plan_id']}/publish",
            json={"content": {"workstreams": []}, "workstream_count": 0},
        )
        other_snapshot = other_publish_resp.json()["snapshot"]

        other_orch_resp = app_request(
            "POST", f"/api/v1/plans/{other_plan['plan_id']}/orchestrate",
            json={"plan_snapshot_id": other_snapshot["plan_snapshot_id"]},
        )
        other_orchestration_id = other_orch_resp.json()["orchestration"]["orchestration_id"]

        resp = app_request(
            "POST",
            f"/api/v1/orchestrations/{other_orchestration_id}/workstreams/{WORKSTREAM_KEY}/dispatch",
            json={"worker_identity": "post-deploy-validator"},
        )
        duration_ms = (time.time() - start) * 1000
        body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {"raw": resp.text}

        if resp.status_code == 404:
            suite.add(TestResult(
                name="negative_scoping_cross_orchestration", status="PASS", duration_ms=duration_ms,
                message=(
                    f"workstream_key '{WORKSTREAM_KEY}' correctly not found under an "
                    "unrelated orchestration_id"
                ),
                diagnostic={"status_code": resp.status_code, "body": body, "other_orchestration_id": other_orchestration_id},
            ))
        else:
            suite.add(TestResult(
                name="negative_scoping_cross_orchestration", status="FAIL", duration_ms=duration_ms,
                message=(
                    f"expected 404, got HTTP {resp.status_code} — dispatch may have matched "
                    "the workstream_key across orchestration boundaries"
                ),
                diagnostic={"status_code": resp.status_code, "body": body, "other_orchestration_id": other_orchestration_id},
            ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="negative_scoping_cross_orchestration", status="FAIL", duration_ms=duration_ms,
        message=f"request failed: {type(e).__name__}: {e}",
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 5: MCP tools/list")
print("─" * 40)

start = time.time()
try:
    resp = app_request(
        "POST", "/mcp",
        json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
        headers={"Content-Type": "application/json"},
    )
    duration_ms = (time.time() - start) * 1000
    body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {"raw": resp.text}

    if resp.status_code != 200:
        suite.add(TestResult(
            name="mcp_tools_list", status="FAIL", duration_ms=duration_ms,
            message=f"POST /mcp tools/list → HTTP {resp.status_code}",
            diagnostic={"status_code": resp.status_code, "body": body},
        ))
    else:
        tools = body.get("result", {}).get("tools", [])
        record_decision = next((t for t in tools if t.get("name") == "record_decision"), None)

        if not tools:
            suite.add(TestResult(
                name="mcp_tools_list", status="FAIL", duration_ms=duration_ms,
                message="tools/list returned zero tools",
                diagnostic={"body": body},
            ))
        elif record_decision is None:
            suite.add(TestResult(
                name="mcp_tools_list", status="FAIL", duration_ms=duration_ms,
                message=f"record_decision tool missing from {len(tools)} registered tools",
                diagnostic={"tool_names": [t.get("name") for t in tools]},
            ))
        else:
            required = record_decision.get("inputSchema", {}).get("required", [])
            if "workstream_id" not in required:
                suite.add(TestResult(
                    name="mcp_tools_list", status="FAIL", duration_ms=duration_ms,
                    message=(
                        "record_decision does not require workstream_id, but state.decisions "
                        "has workstream_id UUID NOT NULL — a write with an omitted workstream_id "
                        "would be accepted by the tool schema and then rejected by the DB"
                    ),
                    diagnostic={"record_decision_input_schema": record_decision.get("inputSchema")},
                ))
            else:
                suite.add(TestResult(
                    name="mcp_tools_list", status="PASS", duration_ms=duration_ms,
                    message=f"{len(tools)} tools registered; record_decision requires workstream_id",
                    diagnostic={"tool_names": [t.get("name") for t in tools]},
                ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="mcp_tools_list", status="FAIL", duration_ms=duration_ms,
        message=f"request failed: {type(e).__name__}: {e}",
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 6: RLS Write Integrity (MCP start_session)")
print("─" * 40)

start = time.time()
try:
    # Call start_session via the MCP JSON-RPC endpoint.
    # After migration 007, state.sessions has RLS with an INSERT policy:
    #   user_id = current_setting('app.user_identity', true)
    # If withIdentity() is wired correctly, the INSERT succeeds and the
    # returned user_id matches the authenticated SPN identity.
    rls_project_id = str(uuid.uuid4())
    rls_workstream_id = str(uuid.uuid4())
    mcp_resp = app_request(
        "POST", "/mcp",
        json={
            "jsonrpc": "2.0",
            "method": "tools/call",
            "id": 2,
            "params": {
                "name": "start_session",
                "arguments": {
                    "project_id": rls_project_id,
                    "workstream_id": rls_workstream_id,
                    "metadata": {"test": "rls_write_integrity"},
                },
            },
        },
        headers={"Content-Type": "application/json"},
    )
    duration_ms = (time.time() - start) * 1000
    body = mcp_resp.json() if mcp_resp.headers.get("content-type", "").startswith("application/json") else {"raw": mcp_resp.text}

    if mcp_resp.status_code != 200:
        suite.add(TestResult(
            name="rls_write_integrity_start_session", status="FAIL", duration_ms=duration_ms,
            message=f"POST /mcp tools/call start_session → HTTP {mcp_resp.status_code}",
            diagnostic={"status_code": mcp_resp.status_code, "body": body},
        ))
    elif "error" in body:
        # JSON-RPC error (e.g., RLS policy violation would surface here)
        suite.add(TestResult(
            name="rls_write_integrity_start_session", status="FAIL", duration_ms=duration_ms,
            message=f"JSON-RPC error: {body['error'].get('message', 'unknown')}",
            diagnostic={"body": body},
        ))
    else:
        # Extract the tool result — MCP tools/call returns content array
        tool_result = body.get("result", {})
        content_items = tool_result.get("content", [])
        text_content = next((c.get("text", "") for c in content_items if c.get("type") == "text"), "")
        try:
            session_data = json.loads(text_content) if text_content else {}
        except json.JSONDecodeError:
            session_data = {"raw_text": text_content}

        returned_user_id = session_data.get("user_id", "")
        session_id = session_data.get("session_id", "")

        if not returned_user_id:
            suite.add(TestResult(
                name="rls_write_integrity_start_session", status="FAIL", duration_ms=duration_ms,
                message="start_session returned no user_id — withIdentity may not be wired",
                diagnostic={"session_data": session_data, "full_body": body},
            ))
        elif returned_user_id == SPN_CLIENT_ID or "@" in returned_user_id:
            # The user_id should be the SPN's identity as extracted from headers.
            # Depending on the auth flow, it may be the client_id UUID or an email.
            suite.add(TestResult(
                name="rls_write_integrity_start_session", status="PASS", duration_ms=duration_ms,
                message=(
                    f"start_session succeeded with RLS active; "
                    f"user_id='{returned_user_id}' session_id='{session_id[:8]}...'"
                ),
                diagnostic={"session_data": session_data, "spn_client_id": SPN_CLIENT_ID},
            ))
        else:
            # user_id is present but doesn't look like a valid identity
            suite.add(TestResult(
                name="rls_write_integrity_start_session", status="FAIL", duration_ms=duration_ms,
                message=(
                    f"user_id '{returned_user_id}' does not match expected SPN identity — "
                    "identity extraction or set_config may be broken"
                ),
                diagnostic={"session_data": session_data, "spn_client_id": SPN_CLIENT_ID},
            ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="rls_write_integrity_start_session", status="FAIL", duration_ms=duration_ms,
        message=f"request failed: {type(e).__name__}: {e}",
    ))

In [0]:
print("\n" + "─" * 40)
print("TEST 7: RLS Policy Audit (direct Lakebase pg_policies + pg_class)")
print("─" * 40)

# ── Direct Lakebase query using psycopg2 ────────────────────────────────────
# The Databricks SDK (w.postgres) is for resource management only — it does NOT
# have an execute_statement/execute_sql method. To query Postgres system catalogs
# (pg_class, pg_policies) we connect directly via psycopg2 with an OAuth token.
import psycopg2

LAKEBASE_DB_PATH = f"projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH}/databases/{LAKEBASE_DATABASE}"

# Resolve the endpoint host from the SDK
_lb_endpoints = list(w.postgres.list_endpoints(
    parent=f"projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH}"
))
_lb_host = _lb_endpoints[0].status.hosts.host if _lb_endpoints else None
print(f"[lakebase] Endpoint host: {_lb_host}")

# Reuse the SPN OAuth token (already acquired in cell 4) for Postgres auth.
# The Lakebase endpoint accepts workspace-scoped OAuth tokens.


def lakebase_query(sql: str) -> list[list]:
    """Execute SQL against Lakebase via psycopg2 and return rows as lists."""
    conn = psycopg2.connect(
        host=_lb_host,
        port=5432,
        dbname=LAKEBASE_DATABASE,
        user=SPN_CLIENT_ID,
        password=SPN_TOKEN,
        sslmode="require",
        options="-c statement_timeout=30000",
    )
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            return [list(row) for row in cur.fetchall()]
    finally:
        conn.close()


# ── Expected RLS configuration (migration 007) ─────────────────────────────
# 8 tables, each with ENABLE + FORCE ROW LEVEL SECURITY
STATE_RLS_TABLES = [
    "sessions", "decisions", "errors", "events",
    "task_attempts", "task_completions", "workstream_events", "recovery_state",
]

# Expected policies: every table gets a select + insert policy;
# mutable tables (sessions, recovery_state) additionally get an update policy.
EXPECTED_POLICIES = {
    "sessions": ["sessions_select_policy", "sessions_insert_policy", "sessions_update_policy"],
    "decisions": ["decisions_select_policy", "decisions_insert_policy"],
    "errors": ["errors_select_policy", "errors_insert_policy"],
    "events": ["events_select_policy", "events_insert_policy"],
    "task_attempts": ["task_attempts_select_policy", "task_attempts_insert_policy"],
    "task_completions": ["task_completions_select_policy", "task_completions_insert_policy"],
    "workstream_events": ["workstream_events_select_policy", "workstream_events_insert_policy"],
    "recovery_state": ["recovery_state_select_policy", "recovery_state_insert_policy", "recovery_state_update_policy"],
}
TOTAL_EXPECTED_POLICIES = sum(len(v) for v in EXPECTED_POLICIES.values())

start = time.time()
try:
    # ── Check 1: RLS enabled + forced on all 8 tables ──────────────────────
    rls_flags_sql = """
        SELECT c.relname AS table_name,
               c.relrowsecurity AS rls_enabled,
               c.relforcerowsecurity AS rls_forced
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = 'state'
          AND c.relkind = 'r'
          AND c.relname IN ('sessions','decisions','errors','events',
                            'task_attempts','task_completions',
                            'workstream_events','recovery_state')
        ORDER BY c.relname;
    """
    rls_rows = lakebase_query(rls_flags_sql)

    # Parse: each row is [table_name, rls_enabled_bool, rls_forced_bool]
    tables_without_rls = []
    tables_without_force = []
    tables_found = set()
    for row in rls_rows:
        tname, enabled, forced = row[0], row[1], row[2]
        tables_found.add(tname)
        # psycopg2 returns native Python booleans from pg_class
        if not enabled:
            tables_without_rls.append(tname)
        if not forced:
            tables_without_force.append(tname)

    missing_tables = set(STATE_RLS_TABLES) - tables_found

    # ── Check 2: All 16+ policies exist ────────────────────────────────────
    policies_sql = """
        SELECT tablename, policyname
        FROM pg_policies
        WHERE schemaname = 'state'
        ORDER BY tablename, policyname;
    """
    policy_rows = lakebase_query(policies_sql)

    # Build actual policy map
    actual_policies: dict[str, list[str]] = {}
    for row in policy_rows:
        tname, pname = row[0], row[1]
        actual_policies.setdefault(tname, []).append(pname)

    # Compare expected vs actual
    missing_policies = []
    for table, expected_list in EXPECTED_POLICIES.items():
        actual_list = actual_policies.get(table, [])
        for policy_name in expected_list:
            if policy_name not in actual_list:
                missing_policies.append(f"{table}.{policy_name}")

    duration_ms = (time.time() - start) * 1000

    # ── Verdict ────────────────────────────────────────────────────────────
    actual_policy_count = sum(len(v) for v in actual_policies.values())
    diagnostic = {
        "db_path": LAKEBASE_DB_PATH,
        "tables_found": sorted(tables_found),
        "missing_tables": sorted(missing_tables),
        "tables_without_rls_enabled": tables_without_rls,
        "tables_without_force_rls": tables_without_force,
        "actual_policy_count": actual_policy_count,
        "expected_policy_count": TOTAL_EXPECTED_POLICIES,
        "missing_policies": missing_policies,
        "actual_policies": actual_policies,
    }

    if missing_tables:
        suite.add(TestResult(
            name="rls_policy_audit_state", status="FAIL", duration_ms=duration_ms,
            message=f"Tables missing from state schema: {sorted(missing_tables)}",
            diagnostic=diagnostic,
        ))
    elif tables_without_rls or tables_without_force:
        suite.add(TestResult(
            name="rls_policy_audit_state", status="FAIL", duration_ms=duration_ms,
            message=(
                f"RLS not fully enabled: "
                f"missing ENABLE on {tables_without_rls or 'none'}, "
                f"missing FORCE on {tables_without_force or 'none'}"
            ),
            diagnostic=diagnostic,
        ))
    elif missing_policies:
        suite.add(TestResult(
            name="rls_policy_audit_state", status="FAIL", duration_ms=duration_ms,
            message=f"{len(missing_policies)} policies missing: {missing_policies[:5]}{'...' if len(missing_policies) > 5 else ''}",
            diagnostic=diagnostic,
        ))
    else:
        suite.add(TestResult(
            name="rls_policy_audit_state", status="PASS", duration_ms=duration_ms,
            message=(
                f"All {len(STATE_RLS_TABLES)} state tables have RLS enabled+forced; "
                f"{actual_policy_count}/{TOTAL_EXPECTED_POLICIES} policies present"
            ),
            diagnostic=diagnostic,
        ))

except Exception as e:
    duration_ms = (time.time() - start) * 1000
    suite.add(TestResult(
        name="rls_policy_audit_state", status="FAIL", duration_ms=duration_ms,
        message=f"Lakebase query failed: {type(e).__name__}: {e}",
        diagnostic={"db_path": LAKEBASE_DB_PATH, "error": str(e)},
    ))

In [0]:
print("\n")

results_json = json.dumps({
    "summary": suite.summary(),
    "results": [
        {
            "name": r.name,
            "status": r.status,
            "duration_ms": r.duration_ms,
            "message": r.message,
            "critical": r.critical,
            "diagnostic": r.diagnostic,
        }
        for r in suite.results
    ],
}, indent=2, default=str)

print(results_json)

# Final assertion — fails the job if any critical test failed. The raised
# message includes the full diagnostic for each critical failure (see
# TestSuite.assert_passed), so job failure notifications carry raw evidence.
suite.assert_passed()